# CymbalGoal AlloyDB Workshop—Module 1 checks, step 3: DMS end to end

**Track A, Module 1 (aldb001) feature checks.** Runs every Module 1 feature that lives between the
legacy SQL Server and AlloyDB, in the lab project, and records what happened: what a student can
assess, what the conversion workspace and Gemini do with our T-SQL, how long the full load takes, how
far behind continuous replication runs, whether the converted code gives the same answers, and the
read pool.

| | |
| :-- | :-- |
| **Runs in** | Colab Enterprise in the lab project, after step 1 (provisioning) and step 2 (`pipeline/05-sqlserver-package.ipynb`) |
| **Leaves behind** | a **running** continuous migration job in the CDC phase: step 4 (Cloud Shell) validates it with DVT, then promotes it |
| **Writes** | raw evidence (entities, converted DDL, job state, comparisons) and this run's handback under `gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop/module1-checks/` |

Every DMS call is a `gcloud` command a lab could show students, except where only the REST API returns
the detail (converted DDL per object, per-table job state). A failed step is recorded and the run goes
on wherever the next step can still mean something.

**Run all.** About 45–75 minutes, most of it the full load. The last cell is the handback.

| | Section |
| :-- | :-- |
| 1 | Configuration |
| 2 | Environment |
| 3 | Assess the source: what a student can see from SQL Server itself |
| 4 | Prepare the destination: database, migration login, privileges |
| 5 | Connection profiles |
| 6 | Conversion workspace with Gemini: seed, convert, what converted, what was flagged |
| 7 | Apply the converted schema |
| 8 | Continuous migration job: full load |
| 9 | Replication lag |
| 10 | Does the converted code give the same answers? |
| 11 | Identity values, write paths, and the student's access |
| 12 | Python connector: primary and read pool; add a read-pool node |
| 13 | Handback |

---
## 1 · Configuration

In [ ]:
SQL_INSTANCE  = "cymbalgoal-legacy-sql"
DB_SOURCE     = "CymbalGoalScouting"
CLUSTER       = "cymbalgoal-cluster"
PRIMARY       = "cymbalgoal-primary"
READ_POOL     = "cymbalgoal-read-pool"
DMS_DB        = "cymbalgoal_scouting"          # destination database; DMS requires the login to share its name
DMS_USER      = "cymbalgoal_scouting"
SECRET_SQL    = "cymbalgoal-sqlserver-password"
SECRET_DMS    = "cymbalgoal-dms-password"

SRC_PROFILE   = "cg-sqlserver-source"
DST_PROFILE   = "cg-alloydb-destination"
WORKSPACE     = "cg-scouting-workspace"
JOB           = "cg-scouting-migration"

GCS_WORK       = "gs://class-demo/alloydb-labs/cymbalgoal-alloydb-workshop"
RESULTS_PREFIX = f"{GCS_WORK}/module1-checks"

FULL_LOAD_TIMEOUT_MIN = 120
CDC_TRIALS            = 5
CDC_POLL_S            = 0.5
SCALE_READ_POOL_TO    = 2      # Task 6: add a node
SMOKE_PLAYER, SMOKE_CLUB, SMOKE_SEASON = 28003, 131, 2014   # Lionel Messi, FC Barcelona
LEAGUE_CHECK = ("GB1", 2023)

---
## 2 · Environment

Finds the step 1 resources, reads both passwords from Secret Manager, records the gcloud version (DMS
flags change between releases), and checks that the DMS egress IPs reached both allowlists.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cloud-sql-python-connector[pytds]",
                "google-cloud-alloydb-connector[pg8000]", "pandas"], check=True)
print("client libraries ready")

In [ ]:
import json, re, statistics, time
from datetime import datetime, timezone
from pathlib import Path

import google.auth
from google.auth.transport.requests import AuthorizedSession

HB, TIMERS, CMDS, EVIDENCE = {}, {}, [], {}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
WORK = Path("/content/module1-checks")
try:
    WORK.mkdir(parents=True, exist_ok=True)
except OSError:
    WORK = Path.home() / "module1-checks"
    WORK.mkdir(parents=True, exist_ok=True)


def hb(section, line=""):
    HB.setdefault(section, []).append(str(line))


def send_back(section):
    print(f"\n📋 SEND BACK — {section}\n" + "-" * 78)
    for line in HB.get(section, []):
        print(line)
    print("-" * 78)


def sh(cmd, check=True, timeout=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd[:200]}\n{(r.stderr or r.stdout).strip()[:1500]}")
    return r.stdout.strip() if check else r


def gcl(cmd, label, timeout=None):
    """Run a gcloud command, time it, keep a log entry WITHOUT the command text (it may hold a password)."""
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    secs = time.time() - t0
    err = (r.stderr or "").strip()
    CMDS.append({"label": label, "rc": r.returncode, "secs": round(secs, 1),
                 "err": err[-600:] if r.returncode else ""})
    print(f"{'✅' if r.returncode == 0 else '❌'} {label}  ({secs:,.1f} s)" + (f"\n   {err[-600:]}" if r.returncode else ""))
    return r, secs


def short_err(r):
    return re.sub(r"\s+", " ", (r.stderr or r.stdout or "").strip())[-300:]


PROJECT = sh("gcloud config get-value project")
ACCOUNT = sh("gcloud config get-value account")
GCLOUD_VERSION = sh("gcloud version --format='value(\"Google Cloud SDK\")'", check=False).stdout.strip()
_cr, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
if hasattr(_cr, "with_quota_project"):
    _cr = _cr.with_quota_project(PROJECT)
API = AuthorizedSession(_cr)

inst = json.loads(sh(f"gcloud sql instances describe {SQL_INSTANCE} --format=json"))
REGION = inst["region"]
CONNECTION_NAME = inst["connectionName"]
SQL_IP = next(a["ipAddress"] for a in inst.get("ipAddresses", []) if a.get("type") == "PRIMARY")
SQL_ALLOWLIST = [n["value"] for n in inst["settings"]["ipConfiguration"].get("authorizedNetworks", [])]

prim = json.loads(sh(f"gcloud alloydb instances describe {PRIMARY} --cluster={CLUSTER} --region={REGION} --format=json"))
PRIMARY_URI = prim["name"]
ALLOYDB_IP = prim.get("publicIpAddress")
ALLOYDB_ALLOWLIST = [n["cidrRange"] for n in prim.get("networkConfig", {}).get("authorizedExternalNetworks", [])]
rp = json.loads(sh(f"gcloud alloydb instances describe {READ_POOL} --cluster={CLUSTER} --region={REGION} --format=json"))
READ_POOL_URI = rp["name"]

SQL_PASSWORD = sh(f"gcloud secrets versions access latest --secret={SECRET_SQL}")
DMS_PASSWORD = sh(f"gcloud secrets versions access latest --secret={SECRET_DMS}")
STUDENT_DB_USER = ACCOUNT   # IAM database users are named by email (service accounts drop .gserviceaccount.com)

S = "2 · Environment"
print(f"project {PROJECT}  region {REGION}  gcloud {GCLOUD_VERSION}")
print(f"SQL Server {SQL_IP}  allowlist {SQL_ALLOWLIST}")
print(f"AlloyDB primary {ALLOYDB_IP}  allowlist {ALLOYDB_ALLOWLIST}")
print(f"read pool {READ_POOL_URI}  nodes {rp.get('readPoolConfig', {}).get('nodeCount')}")
hb(S, f"project {PROJECT}, region {REGION}, gcloud {GCLOUD_VERSION}, run {RUN_ID}")
hb(S, f"Cloud SQL allowlist: {len(SQL_ALLOWLIST)} entries; AlloyDB primary allowlist: {len(ALLOYDB_ALLOWLIST)} entries")
send_back(S)

---
## 3 · Assess the source

What a student can learn about the legacy database before any migration tooling, from SQL Server
itself (Cloud SQL Studio runs these same queries): object inventory, the data types in use, the T-SQL
constructs each module relies on, collation, and CDC status. Step 4 separately tries the Database
Migration Assessment collector from Cloud Shell.

In [ ]:
from google.cloud.sql.connector import Connector

S = "3 · Assess the source"
sql_connector = Connector(quota_project=PROJECT, refresh_strategy="lazy")


def ss_connect(autocommit=True):
    return sql_connector.connect(CONNECTION_NAME, "pytds", user="sqlserver", password=SQL_PASSWORD,
                                 db=DB_SOURCE, autocommit=autocommit)


def ss_rows(conn, sql, params=None):
    cur = conn.cursor()
    cur.execute(sql, params)
    return [list(r) for r in cur.fetchall()]


ss = ss_connect()
INVENTORY = ss_rows(ss, """SELECT o.type_desc, COUNT(*) FROM sys.objects o
                           WHERE o.schema_id = SCHEMA_ID('dbo') AND o.is_ms_shipped = 0 GROUP BY o.type_desc ORDER BY 1""")
TYPES_USED = ss_rows(ss, """SELECT DATA_TYPE, COUNT(*) FROM INFORMATION_SCHEMA.COLUMNS
                            WHERE TABLE_SCHEMA = 'dbo' GROUP BY DATA_TYPE ORDER BY 2 DESC""")
MODULES = ss_rows(ss, """SELECT o.name, o.type_desc, m.definition FROM sys.sql_modules m
                         JOIN sys.objects o ON o.object_id = m.object_id WHERE o.schema_id = SCHEMA_ID('dbo') ORDER BY o.name""")
CONSTRUCTS = {"CURSOR": r"\bCURSOR\b", "temp table": r"#\w+", "table variable": r"DECLARE\s+@\w+\s+TABLE",
              "TOP": r"\bTOP\s*\(", "WITH TIES": r"WITH\s+TIES", "ISNULL": r"\bISNULL\s*\(", "MERGE": r"\bMERGE\b",
              "FOR XML PATH": r"FOR\s+XML\s+PATH", "TRY/CATCH": r"BEGIN\s+TRY", "THROW/RAISERROR": r"\b(THROW|RAISERROR)\b",
              "explicit transaction": r"BEGIN\s+TRANSACTION", "OUTPUT param": r"\bOUTPUT\b",
              "SCOPE_IDENTITY": r"SCOPE_IDENTITY", "@@ system function": r"@@\w+", "+ string concat": r"N'[^']*'\s*\+|\+\s*N'",
              "DATEDIFF/DATEADD": r"\bDATE(DIFF|ADD)\b", "inserted pseudo-table": r"\binserted\b"}
CONSTRUCT_USE = {k: [n for n, _, d in MODULES if re.search(p, d, flags=re.I)] for k, p in CONSTRUCTS.items()}
COLLATION = ss_rows(ss, "SELECT CONVERT(NVARCHAR(128), DATABASEPROPERTYEX(DB_NAME(), 'Collation'))")[0][0]
CDC_TABLES = ss_rows(ss, "SELECT COUNT(*) FROM sys.tables WHERE schema_id = SCHEMA_ID('dbo') AND is_tracked_by_cdc = 1")[0][0]
ROWS_SS = {r[0]: int(r[1]) for r in ss_rows(ss, """SELECT t.name, SUM(p.rows) FROM sys.tables t
                                                   JOIN sys.partitions p ON p.object_id = t.object_id AND p.index_id IN (0, 1)
                                                   WHERE t.schema_id = SCHEMA_ID('dbo') GROUP BY t.name""")}
TABLES = sorted(ROWS_SS)

print("objects:", ", ".join(f"{a} {b}" for a, b in INVENTORY))
print("types:", ", ".join(f"{a} {b}" for a, b in TYPES_USED))
print(f"collation {COLLATION}; CDC on {CDC_TABLES}/{len(TABLES)} tables")
for k, v in CONSTRUCT_USE.items():
    print(f"  {k:<22} {', '.join(v) or '-'}")
hb(S, "objects: " + ", ".join(f"{a} {b}" for a, b in INVENTORY))
hb(S, "column types: " + ", ".join(f"{a} {b}" for a, b in TYPES_USED))
hb(S, f"collation {COLLATION}; CDC on {CDC_TABLES}/{len(TABLES)} tables; rows " + ", ".join(f"{t} {ROWS_SS[t]:,}" for t in TABLES))
hb(S, "constructs found: " + "; ".join(f"{k} ({len(v)})" for k, v in CONSTRUCT_USE.items() if v))
assert CDC_TABLES == len(TABLES), "CDC is not on for every table: run step 2 (05-sqlserver-package) first"
send_back(S)

---
## 4 · Prepare the destination

DMS writes into a database whose **name matches the login it connects as**: here `cymbalgoal_scouting`,
a built-in (password) user Terraform created with `alloydbsuperuser`. This section creates the
database as that user, and tests which identity can give it `REPLICATION`. DMS needs that attribute
when the source has foreign keys or triggers (ours has both), so replicated rows don't re-fire
triggers or trip key checks.

In [ ]:
from google.cloud.alloydb.connector import Connector as AlloyConnector, IPTypes

S = "4 · Destination"
alloy = AlloyConnector(quota_project=PROJECT, refresh_strategy="lazy")


def pg_connect(db, as_student=False, uri=None, autocommit=True):
    kw = dict(user=STUDENT_DB_USER, enable_iam_auth=True) if as_student else dict(user=DMS_USER, password=DMS_PASSWORD)
    c = alloy.connect(uri or PRIMARY_URI, "pg8000", db=db, ip_type=IPTypes.PUBLIC, **kw)
    c.autocommit = autocommit
    return c


def pg_rows(conn, sql, params=None):
    cur = conn.cursor()
    cur.execute(sql, params or ())
    return [list(r) for r in cur.fetchall()] if cur.description else []


def try_sql(conn, sql, params=None):
    try:
        cur = conn.cursor()
        cur.execute(sql, params or ())
        return "OK"
    except Exception as e:  # noqa: BLE001
        try:
            conn.rollback()
        except Exception:  # noqa: BLE001
            pass
        return f"{type(e).__name__}: {str(e)[:200]}"


dms_admin = pg_connect("postgres")
student_admin = pg_connect("postgres", as_student=True)
if not pg_rows(dms_admin, "SELECT 1 FROM pg_database WHERE datname = %s", (DMS_DB,)):
    print("create database:", try_sql(dms_admin, f'CREATE DATABASE "{DMS_DB}"'))
OWNER_DB = pg_rows(dms_admin, "SELECT pg_get_userbyid(datdba) FROM pg_database WHERE datname = %s", (DMS_DB,))[0][0]

REPL = {}
REPL["student (alloydbsuperuser) ALTER ROLE ... REPLICATION"] = try_sql(student_admin, f'ALTER ROLE "{DMS_USER}" WITH REPLICATION')
if REPL["student (alloydbsuperuser) ALTER ROLE ... REPLICATION"] != "OK":
    REPL["dms user on itself"] = try_sql(dms_admin, f'ALTER ROLE "{DMS_USER}" WITH REPLICATION')
ROLES = pg_rows(dms_admin, """SELECT rolname, rolsuper, rolcreaterole, rolcreatedb, rolreplication
                               FROM pg_roles WHERE rolname IN (%s, %s, 'alloydbsuperuser') ORDER BY rolname""",
                (DMS_USER, STUDENT_DB_USER))
for k, v in REPL.items():
    print(f"  {k}: {v}")
print("roles (name, super, createrole, createdb, replication):")
for r in ROLES:
    print("  ", r)
hb(S, f"database {DMS_DB} owned by {OWNER_DB}")
for k, v in REPL.items():
    hb(S, f"{k}: {v}")
hb(S, "roles (super, createrole, createdb, replication): " +
      "; ".join(f"{r[0].split('@')[0]} {r[1:]}" for r in ROLES))
send_back(S)

---
## 5 · Connection profiles

Both profiles use **IP allowlist** connectivity to public IPs: the networking Start Lab can build with
no proxy VM, because DMS's egress IPs are already on both allowlists (section 2).

In [ ]:
S = "5 · Connection profiles"
BASE = f"--region={REGION} --quiet"


def exists(kind, name):
    return subprocess.run(f"gcloud database-migration {kind} describe {name} {BASE} --format='value(name)'",
                          shell=True, capture_output=True, text=True).returncode == 0


PROFILE_RESULTS = {}
if not exists("connection-profiles", SRC_PROFILE):
    r, t = gcl(f"gcloud database-migration connection-profiles create sqlserver {SRC_PROFILE} {BASE} --no-async "
               f"--display-name='CymbalGoal legacy SQL Server' --role=SOURCE --host={SQL_IP} --port=1433 "
               f"--username=sqlserver --password='{SQL_PASSWORD}' --database={DB_SOURCE} --static-ip-connectivity",
               "create source profile (sqlserver, IP allowlist)")
    PROFILE_RESULTS["source"] = ("OK" if r.returncode == 0 else short_err(r), t)
else:
    PROFILE_RESULTS["source"] = ("existed", 0)

if not exists("connection-profiles", DST_PROFILE):
    attempts = [
        ("alloydb-cluster + public host", f"--alloydb-cluster={CLUSTER} --host={ALLOYDB_IP} --port=5432"),
        ("public host only", f"--host={ALLOYDB_IP} --port=5432"),
    ]
    for label, conn_flags in attempts:
        r, t = gcl(f"gcloud database-migration connection-profiles create postgresql {DST_PROFILE} {BASE} --no-async "
                   f"--display-name='CymbalGoal AlloyDB' --role=DESTINATION {conn_flags} --username={DMS_USER} "
                   f"--password='{DMS_PASSWORD}' --database={DMS_DB} --static-ip-connectivity",
                   f"create destination profile ({label})")
        PROFILE_RESULTS["destination"] = ("OK " + label if r.returncode == 0 else short_err(r), t)
        if r.returncode == 0:
            break
else:
    PROFILE_RESULTS["destination"] = ("existed", 0)

for k, (v, t) in PROFILE_RESULTS.items():
    hb(S, f"{k} profile: {v} ({t:,.0f} s)")
send_back(S)

---
## 6 · Conversion workspace with Gemini

Creates the workspace with all four Gemini features on (auto-conversion, conversion assistance,
pattern matching, quality assessment), pulls the source schema, converts it, and then reads every
entity back through the API with its converted DDL and issues. The questions:

- What converted cleanly, and what was flagged, by object?
- How did `MONEY`, `TINYINT`, `BIT`, `DATETIME2` and `IDENTITY` map?
- **Did the conversion keep `usp_search_players` case-insensitive** (`ILIKE`, `lower()`, a
  case-insensitive collation), or is the trap intact?
- Did `usp_record_transfer` keep a `COMMIT` that will fail at run time?
- Did the CDC system schema (`cdc`) come along with `dbo`?

The full DDL of every converted object is saved to the bucket for the lab-writing conversation.

In [ ]:
S = "6 · Conversion"
WS_NAME = f"projects/{PROJECT}/locations/{REGION}/conversionWorkspaces/{WORKSPACE}"
DMS_API = f"https://datamigration.googleapis.com/v1/projects/{PROJECT}/locations/{REGION}"
GEMINI_MODE = "existing workspace"

if not exists("conversion-workspaces", WORKSPACE):
    create = (f"gcloud database-migration conversion-workspaces create {WORKSPACE} {BASE} --no-async "
              f"--display-name='CymbalGoal scouting' --source-database-engine=SQL_SERVER "
              f"--source-database-provider=CLOUDSQL --destination-database-engine=POSTGRESQL "
              f"--destination-database-provider=ALLOYDB")
    for mode, extra in [
        ("gemini flags", "--auto-conversion --conversion-assistance --pattern-matching --quality-assessment"),
        ("global settings", "--global-settings=aiAssistant=true,aiConversion=true,aiPatternApplier=true"),
        ("no gemini", ""),
    ]:
        r, t = gcl(f"{create} {extra}", f"create workspace ({mode})")
        if r.returncode == 0:
            GEMINI_MODE = mode
            TIMERS["create workspace"] = t
            break

r, TIMERS["seed (pull schema)"] = gcl(f"gcloud database-migration conversion-workspaces seed {WORKSPACE} {BASE} "
                                      f"--source-connection-profile={SRC_PROFILE} --auto-commit --no-async",
                                      "seed: pull the source schema")
SEED_OK = r.returncode == 0
r, TIMERS["convert"] = gcl(f"gcloud database-migration conversion-workspaces convert {WORKSPACE} {BASE} "
                           f"--auto-commit --no-async", "convert (all entities)")
CONVERT_OK = r.returncode == 0


def entities(tree):
    out, token = [], None
    while True:
        params = {"tree": tree, "view": "DATABASE_ENTITY_VIEW_FULL", "uncommitted": "true", "pageSize": 1000}
        if token:
            params["pageToken"] = token
        resp = API.get(f"{DMS_API}/conversionWorkspaces/{WORKSPACE}:describeDatabaseEntities", params=params)
        if not resp.ok:
            print(f"describeDatabaseEntities {tree}: {resp.status_code} {resp.text[:300]}")
            return out
        j = resp.json()
        out += j.get("databaseEntities", [])
        token = j.get("nextPageToken")
        if not token:
            return out


SRC_TREE, DRAFT_TREE = entities("SOURCE_TREE"), entities("DRAFT_TREE")
EVIDENCE["source_tree"], EVIDENCE["draft_tree"] = SRC_TREE, DRAFT_TREE
ws = API.get(f"{DMS_API}/conversionWorkspaces/{WORKSPACE}").json()
EVIDENCE["workspace"] = ws


def etype(e):
    return (e.get("entityType") or "?").replace("DATABASE_ENTITY_TYPE_", "")


def counts(ents, key):
    out = {}
    for e in ents:
        k = key(e)
        out[k] = out.get(k, 0) + 1
    return dict(sorted(out.items()))


print("workspace globalSettings:", ws.get("globalSettings"))
print("source tree:", counts(SRC_TREE, lambda e: f"{e.get('parentEntity', '')}/{etype(e)}"))
print("draft tree: ", counts(DRAFT_TREE, lambda e: f"{e.get('parentEntity', '')}/{etype(e)}"))

# ---- issues by object ------------------------------------------------------------------------
ISSUES = []
for e in DRAFT_TREE:
    for i in e.get("issues", []) or []:
        ISSUES.append({"object": f"{e.get('parentEntity', '')}.{e.get('shortName')}", "type": etype(e),
                       "severity": (i.get("severity") or "").replace("ISSUE_SEVERITY_", ""),
                       "kind": (i.get("type") or "").replace("ISSUE_TYPE_", ""), "code": i.get("code"),
                       "message": re.sub(r"\s+", " ", i.get("message") or "")[:220]})
by_sev = counts(ISSUES, lambda i: i["severity"])
print(f"\nissues: {len(ISSUES)}  by severity {by_sev}")
for i in ISSUES:
    if i["severity"] not in ("INFO",):
        print(f"  [{i['severity']}] {i['object']} ({i['type']}): {i['code']} {i['message']}")

# ---- the DDL that would be applied -------------------------------------------------------------
def ddl_of(e):
    return "\n".join(d.get("ddl", "") for d in (e.get("entityDdl") or []))


def ddl_kinds(e):
    return sorted({(d.get("ddlKind") or "?") for d in (e.get("entityDdl") or [])})


DDL = {f"{e.get('parentEntity', '')}.{e.get('shortName')}": ddl_of(e) for e in DRAFT_TREE}
(WORK / f"converted-ddl-{RUN_ID}.sql").write_text(
    "\n\n".join(f"-- ===== {k} ({etype(e)}; ddlKind {ddl_kinds(e)})\n{ddl_of(e)}"
                for k, e in ((f"{e.get('parentEntity', '')}.{e.get('shortName')}", e) for e in DRAFT_TREE)))

In [ ]:
S = "6 · Conversion"
CODE_OBJECTS = ["fn_player_age", "vw_player_profile", "trg_players_modified", "usp_search_players",
                "usp_get_player_season_stats", "usp_club_season_table", "usp_top_scorers",
                "usp_player_career_path", "usp_upsert_valuation", "usp_record_transfer"]


def draft(name):
    return [e for e in DRAFT_TREE if (e.get("shortName") or "").lower() == name.lower()]


print(f"{'object':<30}{'type':<18}{'issues (sev)':<34}ddlKind")
OBJ_SUMMARY = {}
for n in CODE_OBJECTS:
    ents = draft(n)
    if not ents:
        print(f"{n:<30}NOT IN DRAFT TREE")
        OBJ_SUMMARY[n] = "not in draft tree"
        continue
    e = ents[0]
    sev = counts([i for i in ISSUES if i["object"].endswith("." + e.get("shortName"))], lambda i: i["severity"])
    OBJ_SUMMARY[n] = f"{etype(e)}; issues {sev or 'none'}; ddlKind {ddl_kinds(e)}"
    print(f"{n:<30}{etype(e):<18}{str(sev or 'none'):<34}{ddl_kinds(e)}")

# ---- the trap and the watch list --------------------------------------------------------------
def text_of(n):
    return "\n".join(ddl_of(e) for e in draft(n))


search = text_of("usp_search_players")
TRAP = ("CASE HANDLED: " + ", ".join(k for k, p in [("ILIKE", r"\bILIKE\b"), ("lower()/upper()", r"\b(lower|upper)\s*\("),
                                                     ("COLLATE", r"\bCOLLATE\b"), ("citext", r"citext")]
                                     if re.search(p, search, flags=re.I))) \
       if re.search(r"\bILIKE\b|\b(lower|upper)\s*\(|\bCOLLATE\b|citext", search, flags=re.I) else \
       ("TRAP INTACT: plain LIKE" if re.search(r"\bLIKE\b", search, flags=re.I) else "no LIKE found (check the DDL)")
rec = text_of("usp_record_transfer")
WATCH = {
    "usp_search_players case handling": TRAP,
    "usp_record_transfer keeps COMMIT": bool(re.search(r"\bCOMMIT\b", rec, flags=re.I)),
    "usp_record_transfer has EXCEPTION block": bool(re.search(r"\bEXCEPTION\b", rec, flags=re.I)),
    "usp_player_career_path aggregation": ("string_agg" if re.search(r"string_agg", text_of("usp_player_career_path"), re.I)
                                            else "xml" if re.search(r"xml", text_of("usp_player_career_path"), re.I) else "other"),
    "usp_upsert_valuation MERGE": ("MERGE kept" if re.search(r"\bMERGE\b", text_of("usp_upsert_valuation"), re.I)
                                    else "rewritten (no MERGE)"),
    "fn_player_age date math": ", ".join(sorted(set(m.lower() for m in re.findall(
        r"\b(age|date_part|extract|interval|datediff|date_trunc|make_interval)\b", text_of("fn_player_age"), flags=re.I)))) or "?",
}

# ---- type mapping, read off the converted table DDL ---------------------------------------------
def col_type(table, col):
    m = re.search(rf'(?im)^\s*"?{col}"?\s+([^,\n]+)', text_of(table))
    return re.sub(r"\s+", " ", m.group(1)).strip()[:60] if m else "?"


TYPE_MAP = {
    "MONEY (transfers.transfer_fee)": col_type("transfers", "transfer_fee"),
    "TINYINT (appearances.goals)": col_type("appearances", "goals"),
    "BIT (players.is_active)": col_type("players", "is_active"),
    "DATETIME2(3) (players.created_at)": col_type("players", "created_at"),
    "IDENTITY (transfers.transfer_id)": col_type("transfers", "transfer_id"),
    "NVARCHAR(100) (players.player_name)": col_type("players", "player_name"),
    "DECIMAL(4,1) (clubs.average_age)": col_type("clubs", "average_age"),
    "DATE (players.date_of_birth)": col_type("players", "date_of_birth"),
}
CDC_SCHEMA_IN_DRAFT = sum(1 for e in DRAFT_TREE if (e.get("parentEntity") or "").lower() == "cdc")
SCHEMAS_IN_DRAFT = sorted({(e.get("parentEntity") or "") for e in DRAFT_TREE})

print()
for k, v in {**WATCH, **TYPE_MAP}.items():
    print(f"  {k:<40} {v}")
print(f"  draft schemas: {SCHEMAS_IN_DRAFT}; entities under cdc: {CDC_SCHEMA_IN_DRAFT}")

hb(S, f"Gemini setup: {GEMINI_MODE}; globalSettings {ws.get('globalSettings')}")
hb(S, f"seed {'OK' if SEED_OK else 'FAILED'} {TIMERS.get('seed (pull schema)', 0):,.0f} s; "
      f"convert {'OK' if CONVERT_OK else 'FAILED'} {TIMERS.get('convert', 0):,.0f} s")
hb(S, f"source entities {len(SRC_TREE)}, draft entities {len(DRAFT_TREE)}; draft schemas {SCHEMAS_IN_DRAFT}; cdc entities {CDC_SCHEMA_IN_DRAFT}")
hb(S, f"issues {len(ISSUES)} by severity {by_sev}")
for i in ISSUES:
    if i["severity"] not in ("INFO",):
        hb(S, f"  [{i['severity']}] {i['object']}: {i['code']} {i['message'][:160]}")
for n, v in OBJ_SUMMARY.items():
    hb(S, f"  {n}: {v}")
for k, v in {**WATCH, **TYPE_MAP}.items():
    hb(S, f"  {k}: {v}")
send_back(S)

---
## 7 · Apply the converted schema

`apply` creates the converted tables, view, function, trigger and procedures in `cymbalgoal_scouting`.
Afterwards the destination's catalog is read back: which schema the objects landed in, who owns them,
and which routines exist with which signatures.

In [ ]:
S = "7 · Apply"
r, TIMERS["apply"] = gcl(f"gcloud database-migration conversion-workspaces apply {WORKSPACE} {BASE} "
                         f"--destination-connection-profile={DST_PROFILE} --no-async", "apply to AlloyDB")
APPLY_OK = r.returncode == 0
APPLY_ERR = "" if APPLY_OK else short_err(r)

dst = pg_connect(DMS_DB)
DEST_TABLES = pg_rows(dst, """SELECT schemaname, tablename, tableowner FROM pg_tables
                              WHERE schemaname NOT IN ('pg_catalog', 'information_schema') ORDER BY 1, 2""")
SCHEMA = next((s for s, t, o in DEST_TABLES if t == "players"), None)
DEST_OWNER = next((o for s, t, o in DEST_TABLES if t == "players"), None)
ROUTINES = {}
if SCHEMA:
    for name, kind, names, modes, types, result in pg_rows(dst, """
            SELECT p.proname, p.prokind,
                   COALESCE(p.proargnames, ARRAY[]::text[]),
                   COALESCE(p.proargmodes::text[], array_fill('i'::text, ARRAY[p.pronargs::int])),
                   ARRAY(SELECT format_type(t, NULL) FROM unnest(COALESCE(p.proallargtypes, p.proargtypes::oid[])) t),
                   pg_get_function_result(p.oid)
            FROM pg_proc p JOIN pg_namespace n ON n.oid = p.pronamespace
            WHERE n.nspname = %s ORDER BY 1""", (SCHEMA,)):
        ROUTINES.setdefault(name, {"kind": kind, "names": names, "modes": modes, "types": types, "result": result})
TRIGGERS = pg_rows(dst, """SELECT tgname, tgrelid::regclass::text, tgenabled FROM pg_trigger WHERE NOT tgisinternal""")
VIEWS = pg_rows(dst, "SELECT schemaname, viewname FROM pg_views WHERE schemaname = %s", (SCHEMA or "",))
print(f"schema {SCHEMA}, owner {DEST_OWNER}, tables {len([1 for s, t, o in DEST_TABLES if s == SCHEMA])}")
for n, v in ROUTINES.items():
    sig = ", ".join(f"{m}:{a or '?'} {t}" for m, a, t in zip(v["modes"], list(v["names"]) + [""] * 20, v["types"]))
    print(f"  {v['kind']} {n}({sig}) -> {v['result']}")
print("triggers:", TRIGGERS, "\nviews:", VIEWS)

hb(S, f"apply {'OK' if APPLY_OK else 'FAILED: ' + APPLY_ERR} ({TIMERS['apply']:,.0f} s)")
hb(S, f"landed in schema {SCHEMA}, tables owned by {DEST_OWNER}; "
      f"other schemas: {sorted({s for s, t, o in DEST_TABLES} - {SCHEMA})}")
for n, v in ROUTINES.items():
    hb(S, f"  {'procedure' if v['kind'] == 'p' else 'function'} {n}: modes {''.join(v['modes'])} -> {v['result']}")
hb(S, f"triggers {[(t[0], t[1], t[2]) for t in TRIGGERS]}; views {[v[1] for v in VIEWS]}")
send_back(S)

---
## 8 · Continuous migration job: full load

Creates a `CONTINUOUS` job from the two profiles and the workspace, starts it, and polls it every ten
seconds, recording each state and phase change. The full load is done when the phase reaches `CDC`.
Then per-table state comes from the job's objects, and row counts are compared table by table.

In [ ]:
S = "8 · Full load"
JOB_NAME = f"projects/{PROJECT}/locations/{REGION}/migrationJobs/{JOB}"
JOB_CREATE = "existed"
if not exists("migration-jobs", JOB):
    create = (f"gcloud database-migration migration-jobs create {JOB} {BASE} --no-async --type=CONTINUOUS "
              f"--display-name='CymbalGoal scouting to AlloyDB' --source={SRC_PROFILE} --destination={DST_PROFILE} "
              f"--conversion-workspace={WS_NAME}")
    for label, extra in [("with --static-ip", "--static-ip"), ("default connectivity", "")]:
        r, TIMERS["create job"] = gcl(f"{create} {extra}", f"create migration job ({label})")
        if r.returncode == 0:
            JOB_CREATE = label
            break
        JOB_CREATE = "FAILED: " + short_err(r)


def job():
    resp = API.get(f"{DMS_API}/migrationJobs/{JOB}")
    return resp.json() if resp.ok else {"state": f"HTTP {resp.status_code}", "error": {"message": resp.text[:300]}}


j = job()
TRANSITIONS = []
T_START = time.time()
if j.get("state") in ("NOT_STARTED", "DRAFT", "CREATING"):
    r, _ = gcl(f"gcloud database-migration migration-jobs start {JOB} {BASE}", "start migration job")
last = None
while True:
    j = job()
    now = (j.get("state"), j.get("phase"))
    if now != last:
        TRANSITIONS.append((round(time.time() - T_START), *now))
        print(f"  +{TRANSITIONS[-1][0]:>5} s  state {now[0]}  phase {now[1]}")
        last = now
    if j.get("phase") == "CDC" or j.get("state") in ("FAILED", "STOPPED", "COMPLETED") \
            or time.time() - T_START > FULL_LOAD_TIMEOUT_MIN * 60:
        break
    time.sleep(10)
T_CDC = next((t for t, s, p in TRANSITIONS if p == "CDC"), None)
T_RUNNING = next((t for t, s, p in TRANSITIONS if s == "RUNNING"), 0)
JOB_ERROR = (j.get("error") or {}).get("message", "")
EVIDENCE["job"] = j

objs = API.get(f"{DMS_API}/migrationJobs/{JOB}/objects", params={"pageSize": 1000})
OBJECTS = objs.json().get("migrationJobObjects", []) if objs.ok else []
EVIDENCE["job_objects"] = OBJECTS
OBJ_STATES = counts(OBJECTS, lambda o: f"{o.get('state')}/{o.get('phase')}")


def obj_name(o):
    so = o.get("sourceObject", {})
    return f"{so.get('schema', '?')}.{so.get('table', so.get('database', '?'))}"


print("objects:", OBJ_STATES)
for o in OBJECTS:
    if o.get("error"):
        print(f"  ❌ {obj_name(o)}: {o['error'].get('message', '')[:200]}")

ROWS_PG = {}
if SCHEMA:
    dst = pg_connect(DMS_DB)
    for t in TABLES:
        try:
            ROWS_PG[t] = int(pg_rows(dst, f'SELECT count(*) FROM "{SCHEMA}"."{t}"')[0][0])
        except Exception as e:  # noqa: BLE001
            ROWS_PG[t] = f"{type(e).__name__}"
            dst = pg_connect(DMS_DB)
ROWS_SS = {r[0]: int(r[1]) for r in ss_rows(ss, """SELECT t.name, SUM(p.rows) FROM sys.tables t
                                                   JOIN sys.partitions p ON p.object_id = t.object_id AND p.index_id IN (0, 1)
                                                   WHERE t.schema_id = SCHEMA_ID('dbo') GROUP BY t.name""")}
for t in TABLES:
    print(f"  {t:<20}{ROWS_SS[t]:>10,}{str(ROWS_PG.get(t)):>12}  {'✅' if ROWS_SS[t] == ROWS_PG.get(t) else '❌'}")

hb(S, f"job create: {JOB_CREATE}")
hb(S, "transitions (s from start, state, phase): " + "; ".join(f"+{t} {s}/{p}" for t, s, p in TRANSITIONS))
hb(S, f"full load: {'reached CDC at +' + str(T_CDC) + ' s' if T_CDC is not None else 'did NOT reach CDC'}"
      + (f"; error {JOB_ERROR[:300]}" if JOB_ERROR else ""))
hb(S, f"job objects: {OBJ_STATES}")
for o in OBJECTS:
    if o.get("error"):
        hb(S, f"  ❌ {obj_name(o)}: {o['error'].get('message', '')[:200]}")
hb(S, "rows SQL Server vs AlloyDB: " + ", ".join(f"{t} {ROWS_SS[t]:,}/{ROWS_PG.get(t)}" for t in TABLES))
send_back(S)

---
## 9 · Replication lag

Each trial changes a row on SQL Server and polls AlloyDB every half second until the change is
visible. Two kinds of change per trial: a new valuation through `usp_upsert_valuation` (an INSERT,
plus an UPDATE to `players` that fires the trigger), and a plain `UPDATE` to one player. The trigger
check compares `players.modified_at` on both sides: equal means the replicated row arrived as stamped
and the converted trigger did not stamp it a second time.

In [ ]:
S = "9 · Replication lag"
LAGS = {"insert (usp_upsert_valuation)": [], "update (players)": []}
TRIGGER = []
if T_CDC is not None and SCHEMA:
    dst = pg_connect(DMS_DB)

    def wait_for(sql, params, timeout=300):
        t0 = time.time()
        while time.time() - t0 < timeout:
            if pg_rows(dst, sql, params):
                return time.time()
            time.sleep(CDC_POLL_S)
        return None

    for i in range(CDC_TRIALS):
        vdate, value, agent = f"2099-{i + 1:02d}-01", 50_000_000 + i, f"cdc-probe-{RUN_ID}-{i}"
        cur = ss.cursor()
        cur.execute(f"DECLARE @a NVARCHAR(10); EXEC dbo.usp_upsert_valuation {SMOKE_PLAYER}, '{vdate}', {value}, NULL, @a OUTPUT;")
        t_commit = time.time()
        t_seen = wait_for(f'SELECT 1 FROM "{SCHEMA}".player_valuations WHERE player_id = %s AND valuation_date = %s::date',
                          (SMOKE_PLAYER, vdate))
        LAGS["insert (usp_upsert_valuation)"].append(None if t_seen is None else t_seen - t_commit)

        cur.execute(f"UPDATE dbo.players SET agent_name = N'{agent}' WHERE player_id = {SMOKE_PLAYER}")
        t_commit = time.time()
        t_seen = wait_for(f'SELECT 1 FROM "{SCHEMA}".players WHERE player_id = %s AND agent_name = %s', (SMOKE_PLAYER, agent))
        LAGS["update (players)"].append(None if t_seen is None else t_seen - t_commit)
        print(f"  trial {i + 1}: insert {LAGS['insert (usp_upsert_valuation)'][-1]}, update {LAGS['update (players)'][-1]}")

    time.sleep(2)
    ss_mod = ss_rows(ss, f"SELECT CONVERT(NVARCHAR(30), modified_at, 126) FROM dbo.players WHERE player_id = {SMOKE_PLAYER}")[0][0]
    pg_mod = pg_rows(dst, f'SELECT modified_at::text FROM "{SCHEMA}".players WHERE player_id = %s', (SMOKE_PLAYER,))[0][0]
    TRIGGER = [ss_mod, pg_mod]


def lag_summary(v):
    got = [x for x in v if x is not None]
    if not got:
        return "no change arrived"
    return (f"median {statistics.median(got):.1f} s, max {max(got):.1f} s, min {min(got):.1f} s "
            f"({len(got)}/{len(v)} arrived)")


for k, v in LAGS.items():
    print(f"{k}: {lag_summary(v)}")
    hb(S, f"{k}: {lag_summary(v)}")
if TRIGGER:
    same = TRIGGER[0][:19] == str(TRIGGER[1]).replace(" ", "T")[:19]
    print(f"modified_at SQL Server {TRIGGER[0]} / AlloyDB {TRIGGER[1]}")
    hb(S, f"trigger: modified_at SQL Server {TRIGGER[0]} / AlloyDB {TRIGGER[1]} -> "
          f"{'same: no double stamp' if same else 'DIFFERENT: the destination trigger fired again'}")
if T_CDC is None:
    hb(S, "skipped: the job never reached CDC")
send_back(S)

---
## 10 · Does the converted code give the same answers?

The smoke-test calls from step 2 run now against **both** databases, on the same (replicated) data.
On AlloyDB each converted routine is called the way its catalog signature says: a function returning
rows with `SELECT * FROM`, a procedure with `CALL` (output parameters passed as `NULL`, result sets
read from any `refcursor` it returns). Values are compared after light normalization (numbers to two
decimals, text trimmed), row by row for the first ten rows.

In [ ]:
S = "10 · Equivalence"
CALLS = {
    # name: (T-SQL, (routine, [inputs]) or a SQL string for AlloyDB)
    "search 'messi'":   ("EXEC dbo.usp_search_players @name_pattern = N'messi';", ("usp_search_players", ["messi", None, 25])),
    "search 'Messi'":   ("EXEC dbo.usp_search_players @name_pattern = N'Messi';", ("usp_search_players", ["Messi", None, 25])),
    "season stats":     (f"DECLARE @g INT, @m INT; EXEC dbo.usp_get_player_season_stats {SMOKE_PLAYER}, {SMOKE_SEASON}, @g OUTPUT, @m OUTPUT; SELECT @g, @m;",
                         ("usp_get_player_season_stats", [SMOKE_PLAYER, SMOKE_SEASON])),
    "league table":     (f"EXEC dbo.usp_club_season_table N'{LEAGUE_CHECK[0]}', {LEAGUE_CHECK[1]};",
                         ("usp_club_season_table", [LEAGUE_CHECK[0], LEAGUE_CHECK[1]])),
    "top scorers":      (f"EXEC dbo.usp_top_scorers N'{LEAGUE_CHECK[0]}', {LEAGUE_CHECK[1]}, 10;",
                         ("usp_top_scorers", [LEAGUE_CHECK[0], LEAGUE_CHECK[1], 10])),
    "career path":      (f"DECLARE @p NVARCHAR(4000); EXEC dbo.usp_player_career_path {SMOKE_PLAYER}, @p OUTPUT; SELECT @p;",
                         ("usp_player_career_path", [SMOKE_PLAYER])),
    "profile view":     (f"SELECT * FROM dbo.vw_player_profile WHERE player_id = {SMOKE_PLAYER};",
                         f'SELECT * FROM "{SCHEMA}".vw_player_profile WHERE player_id = {SMOKE_PLAYER}'),
    "age, day before":  ("SELECT dbo.fn_player_age('1987-06-24', '2026-06-23');",
                         f"SELECT \"{SCHEMA}\".fn_player_age('1987-06-24'::date, '2026-06-23'::date)"),
    "age, birthday":    ("SELECT dbo.fn_player_age('1987-06-24', '2026-06-24');",
                         f"SELECT \"{SCHEMA}\".fn_player_age('1987-06-24'::date, '2026-06-24'::date)"),
}


def ss_call(sql):
    cur = ss.cursor()
    out = []
    try:
        cur.execute(sql)
        while True:
            if cur.description:
                out.append([list(r) for r in cur.fetchall()])
            if not cur.nextset():
                break
        return out, None
    except Exception as e:  # noqa: BLE001
        return out, f"{type(e).__name__}: {str(e)[:200]}"


def pg_call(spec):
    """Returns (list of result sets, error). Runs in a transaction that is rolled back."""
    conn = pg_connect(DMS_DB, autocommit=False)
    cur = conn.cursor()
    try:
        if isinstance(spec, str):
            cur.execute(spec)
            return [[list(r) for r in cur.fetchall()]], None
        name, inputs = spec
        info = ROUTINES.get(name)
        if not info:
            return [], f"routine {name} not found in {SCHEMA}"
        args, params, it = [], [], iter(inputs)
        for mode, typ in zip(info["modes"], info["types"]):
            if mode == "i":
                v = next(it, None)
                args.append(f"CAST(%s AS {typ})"); params.append(None if v is None else str(v))
            elif mode == "b" or (mode == "o" and info["kind"] == "p"):
                args.append(f"NULL::{typ}")
            elif mode == "v":
                args.append(f"NULL::{typ}")
        target = f'"{SCHEMA}"."{name}"({", ".join(args)})'
        sets = []
        res = info["result"] or ""
        if info["kind"] == "p":
            cur.execute(f"CALL {target}", tuple(params))
            row = cur.fetchone() if cur.description else None
            vals = list(row) if row else []
            out_types = [t for m, t in zip(info["modes"], info["types"]) if m in ("b", "o")]
            outs = [v for v, t in zip(vals, out_types) if t != "refcursor"]
            for v, t in zip(vals, out_types):
                if t == "refcursor" and v:
                    cur.execute(f'FETCH ALL FROM "{v}"')
                    sets.append([list(r) for r in cur.fetchall()])
            if outs:
                sets.append([outs])
        elif res.startswith(("SETOF", "TABLE")):
            cur.execute(f"SELECT * FROM {target}", tuple(params))
            sets.append([list(r) for r in cur.fetchall()])
        elif "refcursor" in res:
            cur.execute(f"SELECT {target}", tuple(params))
            for (cname,) in cur.fetchall():
                cur.execute(f'FETCH ALL FROM "{cname}"')
                sets.append([list(r) for r in cur.fetchall()])
        else:
            cur.execute(f"SELECT {target}", tuple(params))
            sets.append([list(r) for r in cur.fetchall()])
        return sets, None
    except Exception as e:  # noqa: BLE001
        return [], f"{type(e).__name__}: {str(e)[:240]}"
    finally:
        try:
            conn.rollback(); conn.close()
        except Exception:  # noqa: BLE001
            pass


def norm(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return int(v)
    s = str(v).strip()
    if s in ("True", "False"):
        return int(s == "True")
    try:
        return round(float(s.replace(",", "").replace("$", "").replace("€", "")), 2)
    except ValueError:
        return s


def compare(a_sets, b_sets):
    if not a_sets and not b_sets:
        return "both empty"
    if len(a_sets) != len(b_sets):
        note = f"{len(a_sets)} result sets vs {len(b_sets)}; "
    else:
        note = ""
    for k, (a, b) in enumerate(zip(a_sets, b_sets)):
        if len(a) != len(b):
            return note + f"set {k + 1}: {len(a)} rows vs {len(b)}"
        for i, (ra, rb) in enumerate(zip(a[:10], b[:10])):
            na, nb = [norm(x) for x in ra], [norm(x) for x in rb]
            if na != nb:
                return note + f"set {k + 1} row {i + 1}: {na[:6]} vs {nb[:6]}"
    return (note + "same") if note else "same"


EQUIV = {}
for label, (tsql, spec) in CALLS.items():
    a, ea = ss_call(tsql)
    b, eb = pg_call(spec) if SCHEMA else ([], "no destination schema")
    verdict = ("SQL Server error " + ea) if ea else ("AlloyDB error " + eb) if eb else compare(a, b)
    EQUIV[label] = {"sqlserver": a, "alloydb": b, "verdict": verdict}
    print(f"  {'✅' if verdict.startswith('same') else '❌'} {label:<18} {verdict[:200]}")
    hb(S, f"{'SAME' if verdict.startswith('same') else 'DIFF'} {label}: {verdict[:220]}")
EVIDENCE["equivalence"] = EQUIV
send_back(S)

---
## 11 · Identity values, write paths, and the student's access

- **Identity values.** DMS copies the rows, but an identity column's sequence may still start at 1.
  The sequence behind `transfers.transfer_id` and `player_valuations.valuation_id` is compared with
  the largest value loaded.
- **Write paths**, in a transaction that is rolled back: `usp_upsert_valuation`, `usp_record_transfer`
  (does its converted `COMMIT` survive a `CALL`?), and a plain `INSERT` into `transfers`.
- **The student's access.** The migrated tables belong to the DMS login. Can the student's IAM user
  read them, and which grant fixes it?

In [ ]:
S = "11 · Identity, writes, access"
IDENT, WRITES, ACCESS = {}, {}, {}
STUDENT_CAN_READ = False
if SCHEMA:
    dst = pg_connect(DMS_DB)
    for table, col in [("transfers", "transfer_id"), ("player_valuations", "valuation_id")]:
        seq = pg_rows(dst, "SELECT pg_get_serial_sequence(%s, %s)", (f'"{SCHEMA}".{table}', col))[0][0]
        mx = pg_rows(dst, f'SELECT max({col}) FROM "{SCHEMA}".{table}')[0][0]
        if seq:
            last, called = pg_rows(dst, f"SELECT last_value, is_called FROM {seq}")[0]
            IDENT[f"{table}.{col}"] = f"sequence {seq} last_value {last} (is_called {called}); max loaded {mx}" + \
                                      ("  -> NEXT INSERT COLLIDES" if (last or 0) < (mx or 0) else "  -> in step")
        else:
            IDENT[f"{table}.{col}"] = f"no sequence (column default: " + str(pg_rows(dst, """
                SELECT column_default FROM information_schema.columns
                WHERE table_schema = %s AND table_name = %s AND column_name = %s""", (SCHEMA, table, col))) + f"); max {mx}"

    WRITES["CALL usp_upsert_valuation"] = pg_call(("usp_upsert_valuation", [SMOKE_PLAYER, "2099-12-31", 1, None]))[1] or "OK"
    WRITES["CALL usp_record_transfer"] = pg_call(("usp_record_transfer", [SMOKE_PLAYER, SMOKE_CLUB, "2099-07-01", 0]))[1] or "OK"
    conn = pg_connect(DMS_DB, autocommit=False)
    try:
        cur = conn.cursor()
        cur.execute(f"""INSERT INTO "{SCHEMA}".transfers (player_id, transfer_date, to_club_id, to_club_name)
                        VALUES (%s, '2099-07-02', %s, 'probe') RETURNING transfer_id""", (SMOKE_PLAYER, SMOKE_CLUB))
        WRITES["INSERT into transfers (default id)"] = f"OK, got transfer_id {cur.fetchone()[0]}"
    except Exception as e:  # noqa: BLE001
        WRITES["INSERT into transfers (default id)"] = f"{type(e).__name__}: {str(e)[:200]}"
    finally:
        conn.rollback(); conn.close()

    # ---- the student's access to tables the DMS login owns ---------------------------------------
    def student_read():
        try:
            c = pg_connect(DMS_DB, as_student=True)
            n = pg_rows(c, f'SELECT count(*) FROM "{SCHEMA}".players')[0][0]
            c.close()
            return f"OK ({n:,} players)"
        except Exception as e:  # noqa: BLE001
            return f"{type(e).__name__}: {str(e)[:160]}"

    ACCESS["student reads before any grant"] = student_read()
    student = pg_connect(DMS_DB, as_student=True)
    ACCESS["student: GRANT cymbalgoal_scouting TO self"] = try_sql(student, f'GRANT "{DMS_USER}" TO "{STUDENT_DB_USER}"')
    student.close()
    ACCESS["student reads after role grant attempt"] = student_read()
    if not ACCESS["student reads after role grant attempt"].startswith("OK"):
        owner = pg_connect(DMS_DB)
        ACCESS["owner grants schema usage + table/routine privileges"] = "; ".join(filter(None, [
            try_sql(owner, f'GRANT USAGE ON SCHEMA "{SCHEMA}" TO "{STUDENT_DB_USER}"'),
            try_sql(owner, f'GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA "{SCHEMA}" TO "{STUDENT_DB_USER}"'),
            try_sql(owner, f'GRANT EXECUTE ON ALL ROUTINES IN SCHEMA "{SCHEMA}" TO "{STUDENT_DB_USER}"'),
        ]))
        ACCESS["student reads after owner grants"] = student_read()
    STUDENT_CAN_READ = any(k.startswith("student reads") and v.startswith("OK") for k, v in ACCESS.items())

for d in (IDENT, WRITES, ACCESS):
    for k, v in d.items():
        print(f"  {k}: {v}")
        hb(S, f"{k}: {v}")
send_back(S)

---
## 12 · Python connector: primary and read pool; add a read-pool node

The AlloyDB Python Connector with IAM auth, as the student: connect time and a typical read (the
top-scorers query) on the primary and on the read pool's instance URI, with `pg_is_in_recovery()`
showing which one answered. Then the read pool goes from one node to two (lab Task 6), timed, and the
read is repeated.

In [ ]:
S = "12 · Connector and read pool"
TOP_SQL = f"""SELECT p.player_name, sum(a.goals) AS goals
              FROM "{SCHEMA}".appearances a JOIN "{SCHEMA}".games g ON g.game_id = a.game_id
              JOIN "{SCHEMA}".players p ON p.player_id = a.player_id
              WHERE g.competition_id = 'GB1' AND g.season = 2023
              GROUP BY p.player_name ORDER BY goals DESC LIMIT 10"""


def probe(uri, label, n=7):
    t0 = time.time()
    c = pg_connect(DMS_DB, as_student=True, uri=uri)
    connect_s = time.time() - t0
    rec = pg_rows(c, "SELECT pg_is_in_recovery()")[0][0]
    times, top = [], None
    for _ in range(n):
        t1 = time.time()
        rows = pg_rows(c, TOP_SQL)
        times.append((time.time() - t1) * 1000)
        top = rows[0] if rows else None
    c.close()
    return {"label": label, "connect_s": round(connect_s, 2), "in_recovery": rec,
            "median_ms": round(statistics.median(times), 1), "top": top}


CONN = []
if SCHEMA and STUDENT_CAN_READ:
    for uri, label in [(PRIMARY_URI, "primary"), (READ_POOL_URI, "read pool, 1 node")]:
        try:
            CONN.append(probe(uri, label))
        except Exception as e:  # noqa: BLE001
            CONN.append({"label": label, "error": f"{type(e).__name__}: {str(e)[:200]}"})

r, TIMERS["read pool 1 -> 2 nodes"] = gcl(
    f"gcloud alloydb instances update {READ_POOL} --cluster={CLUSTER} --region={REGION} "
    f"--read-pool-node-count={SCALE_READ_POOL_TO} --quiet", f"scale read pool to {SCALE_READ_POOL_TO} nodes")
SCALE_RESULT = "OK" if r.returncode == 0 else short_err(r)
NODES_AFTER = sh(f"gcloud alloydb instances describe {READ_POOL} --cluster={CLUSTER} --region={REGION} "
                 f"--format='value(readPoolConfig.nodeCount)'", check=False).stdout.strip()
if SCHEMA and CONN:
    try:
        CONN.append(probe(READ_POOL_URI, f"read pool, {NODES_AFTER} nodes"))
    except Exception as e:  # noqa: BLE001
        CONN.append({"label": f"read pool, {NODES_AFTER} nodes", "error": f"{type(e).__name__}: {str(e)[:200]}"})

for c in CONN:
    print(c)
    hb(S, f"{c['label']}: " + (c["error"] if "error" in c else
       f"connect {c['connect_s']} s, in_recovery {c['in_recovery']}, top-scorers median {c['median_ms']} ms, top {c['top']}"))
hb(S, f"scale read pool to {SCALE_READ_POOL_TO}: {SCALE_RESULT} in {TIMERS['read pool 1 -> 2 nodes']:,.0f} s; nodes now {NODES_AFTER}")
send_back(S)

---
## 13 · Handback

Every SEND BACK block, the gcloud command log (labels, exit codes, times; no command text, since some
carry passwords), and where the raw evidence was saved. The migration job is **left running in CDC**
for step 4. Copy the whole output of this cell and paste it back.

In [ ]:
EVID = WORK / f"step3-evidence-{RUN_ID}.json"
EVID.write_text(json.dumps(EVIDENCE, indent=1, default=str))
saved = []
for f in (EVID, WORK / f"converted-ddl-{RUN_ID}.sql"):
    if f.exists():
        r = subprocess.run(["gcloud", "storage", "cp", str(f), f"{RESULTS_PREFIX}/{f.name}"], capture_output=True, text=True)
        saved.append(f"{f.name}: {'saved' if r.returncode == 0 else 'NOT saved (' + r.stderr.strip()[:120] + ')'}")

lines = ["=" * 78, f"MODULE 1 CHECKS: STEP 3 DMS HANDBACK  (step3-dms-checks.ipynb)  run {RUN_ID}", "=" * 78]
for sec, ls_ in HB.items():
    lines += ["", f"## {sec}", *ls_]
lines += ["", "## gcloud command log (label, exit, seconds, error tail)"]
lines += [f"{c['label']}: rc {c['rc']}, {c['secs']} s" + (f"  ERR {c['err'][-240:]}" if c["rc"] else "") for c in CMDS]
lines += ["", "## Timers", *[f"{k}: {v:,.1f} s" for k, v in TIMERS.items()]]
lines += ["", "## Evidence", *saved, f"migration job {JOB} left running (state {job().get('state')}, phase {job().get('phase')})", "=" * 78]
text = "\n".join(lines)
print(text)
(WORK / "step3-handback.txt").write_text(text)
r = subprocess.run(["gcloud", "storage", "cp", str(WORK / "step3-handback.txt"),
                    f"{RESULTS_PREFIX}/step3-handback-{RUN_ID}.txt"], capture_output=True, text=True)
print(f"\n[saved to {RESULTS_PREFIX}/]" if r.returncode == 0 else f"\n[not saved: {r.stderr.strip()[:200]}]")